In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import torch
from tqdm import tqdm

current_dir = Path.cwd().resolve()
project_root = None
for parent in [current_dir, *current_dir.parents]:
    if (parent / 'src').exists() or (parent / 'requirements.txt').exists():
        project_root = str(parent)
        break
if project_root is None:
    project_root = str(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils.setup import setup_device
from src.utils.dataloader import BreastMNISTLoader

from src.models.main.models import (
    QuantumModel,
    QuantumModelWithEncoder,
    ExtentedQuantumModel,
    ExtentedQuantumModelWithEncoder,
)

from src.models.main.classical import (
    ClassicalModel,
    ShallowCNN,
)

from src.models.extraction import (
    create_extractor,
)


In [ ]:
EXPERIMENT_CONFIG ={
"dataset":"INbreast",
"backbone":"shallow",
"quantum_variant":"basic",
"num_classes":2 ,
"image_size":28 ,
"batch_size":8 ,
"n_qubits":4 ,
"n_layers":1 ,
"target_feature_dim":128 ,
"output_root":"experiments/features",
}

MODEL_REGISTRY ={
"BreastMNIST":{
"classical":{
"resnet":"../experiments/models/classical/BreastMNIST/resnet/best_resnet.pth",
"shallow":"../experiments/models/classical/BreastMNIST/shallow/best_shallow.pth",
},
"quantum":{
"basic":"../experiments/models/quantum/basic/BreastMNIST/checkpoints/model_epoch16.pth",
"trainable":"../experiments/models/quantum/trainable/BreastMNIST/checkpoints/model_epoch15.pth",
},
},
"BUSUCLM":{
"classical":{
"resnet":"../experiments/models/classical/BUSUCLM/resnet/best_resnet.pth",
"shallow":"../experiments/models/classical/BUSUCLM/shallow/best_shallow.pth",
},
"quantum":{
"basic":"../experiments/models/quantum/basic/BUSUCLM/checkpoints/model_epoch18.pth",
"trainable":"../experiments/models/quantum/trainable/BUSUCLM/checkpoints/model_epoch5.pth",
},
},
"INbreast":{
"classical":{
"resnet":"../experiments/models/classical/INbreast/resnet/best_resnet.pth",
"shallow":"../experiments/models/classical/INbreast/shallow/best_shallow.pth",
},
"quantum":{
"basic":"../experiments/models/quantum/basic/INbreast/checkpoints/model_epoch4.pth",
"trainable":"../experiments/models/quantum/trainable/INbreast/checkpoints/model_epoch20.pth",
},
},
}

def _resolve_experiment_paths(config):
    dataset =config ["dataset"]
    backbone =config ["backbone"]
    quantum_variant =config ["quantum_variant"]

    if dataset not in MODEL_REGISTRY :
        raise ValueError(f"Dataset inválido: {dataset }. Opções: {list(MODEL_REGISTRY.keys())}")
    if backbone not in MODEL_REGISTRY [dataset ]["classical"]:
        raise ValueError(f"Backbone inválido: {backbone }.")
    if quantum_variant not in MODEL_REGISTRY [dataset ]["quantum"]:
        raise ValueError(f"Quantum variant inválido: {quantum_variant }.")

    classical_model_path =MODEL_REGISTRY [dataset ]["classical"][backbone ]
    quantum_model_path =MODEL_REGISTRY [dataset ]["quantum"][quantum_variant ]
    output_dir =os.path.join(
    config ["output_root"],
    dataset ,
    f"classical_{backbone }__quantum_{quantum_variant }",
   )

    return classical_model_path, quantum_model_path, output_dir 

CLASSICAL_MODEL_PATH ,QUANTUM_MODEL_PATH ,OUTPUT_DIR =_resolve_experiment_paths(EXPERIMENT_CONFIG)

NUM_CLASSES =EXPERIMENT_CONFIG ["num_classes"]
IMAGE_SIZE =EXPERIMENT_CONFIG ["image_size"]
BATCH_SIZE =EXPERIMENT_CONFIG ["batch_size"]
N_QUBITS =EXPERIMENT_CONFIG ["n_qubits"]
N_LAYERS =EXPERIMENT_CONFIG ["n_layers"]
TARGET_FEATURE_DIM =EXPERIMENT_CONFIG ["target_feature_dim"]

print("Active configuration:")
for k ,v in EXPERIMENT_CONFIG.items():
    print(f"  - {k }: {v }")
print("\nResolved checkpoints:")
print(f"  - Classical: {CLASSICAL_MODEL_PATH }")
print(f"  - Quantum:   {QUANTUM_MODEL_PATH }")
print(f"  - Output:    {OUTPUT_DIR }")

In [ ]:
device =setup_device ()
print (f"Using device: {device }")


def detect_quantum_model_type (checkpoint_path):
    
    checkpoint =torch.load (checkpoint_path ,map_location ='cpu',weights_only =False)

    if isinstance (checkpoint ,dict):
        for key in ("model_state_dict","state_dict","model"):
            if key in checkpoint :
                checkpoint =checkpoint [key ]
                break 

    keys =set (checkpoint.keys ())
    print (f"  First checkpoint keys: {list (keys)[:5 ]}")

    if any ('quanv.'in k for k in keys):
        return "ExtentedQuantumModel"
    elif 'fc.weight'in keys and not any ('quanv.'in k for k in keys):
        return "QuantumModel"
    else :
        print (f"Could not detect the type. Keys: {list (keys)[:10 ]}")
        return None 


def build_classical_model (backbone ,num_classes):
    if backbone =="resnet":
        return ClassicalModel (num_classes =num_classes )
    if backbone =="shallow":
        return ShallowCNN (num_classes =num_classes )
    raise ValueError (f"Backbone clássico não suportado: {backbone }")

def build_quantum_model (checkpoint_path ,n_qubits ,n_layers ,num_classes):
    print ("\nDetecting quantum model type...")
    quantum_type =detect_quantum_model_type (checkpoint_path )
    print (f"  → Detected: {quantum_type }")

    if quantum_type =="ExtentedQuantumModel":
        model =ExtentedQuantumModel (
        n_qubits =n_qubits ,
        n_layers =n_layers ,
        num_classes =num_classes ,
        patch_size =2 ,
        mode ="2d",
        )
        print ("  ✓ Created ExtentedQuantumModel (trainable)")
        return model 

    if quantum_type =="QuantumModel":
        rand_params =np.random.uniform (low =0 ,high =2 *np.pi ,size =(n_layers ,n_qubits ))
        print (f"  Random parameters generated: shape={rand_params.shape }")
        model =QuantumModel (
        n_qubits =n_qubits ,
        n_layers =n_layers ,
        rand_params =rand_params ,
        num_classes =num_classes ,
        patch_size =2 ,
        )
        print ("  ✓ Created QuantumModel (non-trainable)")
        return model 

    raise ValueError ("❌ Could not detect quantum model type!")

def _load_model_checkpoint (model ,checkpoint_path ,device ):
    checkpoint =torch.load (checkpoint_path ,map_location =device ,weights_only =False )

    if isinstance (checkpoint ,dict ):
        for key in ("model_state_dict","state_dict","model"):
            if key in checkpoint :
                checkpoint =checkpoint [key ]
                break 

    if isinstance (checkpoint ,dict ):
        checkpoint ={ 
        (k.replace ("module.","",1 )if k.startswith ("module.")else k ):v 
        for k ,v in checkpoint.items ()
        }

    if hasattr (model ,'quanv'):
        print (f"  Initializing trainable quantum model...")
        model.to (device )
        with torch.no_grad ():
            dummy_input =torch.randn (1 ,1 ,IMAGE_SIZE ,IMAGE_SIZE ).to (device )
            _ =model (dummy_input )

        print (f"  Loading checkpoint weights (FC only)...")
        fc_state ={k :v for k ,v in checkpoint.items ()if k.startswith ('fc.')}
        if fc_state :
            model.load_state_dict (fc_state ,strict =False )
            print (f"  ✓ Loaded FC weights: {list (fc_state.keys ())}")
        else :
            print (f"  No FC weights found in checkpoint")
        return model 

    if hasattr (model ,'fc')and model.fc is None :
        print (f"  Initializing dynamic layers of {type (model ).__name__ }...")
        model.to (device )
        with torch.no_grad ():
            dummy_input =torch.randn (1 ,1 ,IMAGE_SIZE ,IMAGE_SIZE ).to (device )
            try :
                _ =model (dummy_input )
            except Exception as e :
                print (f"  Warning during initialization: {e }")

    missing_keys ,unexpected_keys =model.load_state_dict (checkpoint ,strict =False )
    if missing_keys :
        print (f"  Missing keys: {missing_keys }")
    if unexpected_keys :
        print (f"  Unexpected keys (ignored): {unexpected_keys }")

    return model 

classical_model = build_classical_model(EXPERIMENT_CONFIG["backbone"], NUM_CLASSES)
quantum_model = build_quantum_model(QUANTUM_MODEL_PATH, N_QUBITS, N_LAYERS, NUM_CLASSES)

_load_model_checkpoint(quantum_model, QUANTUM_MODEL_PATH, device)
_load_model_checkpoint(classical_model, CLASSICAL_MODEL_PATH, device)

quantum_model.to(device).eval()
classical_model.to(device).eval()

print(f"  - Quantum: {type(quantum_model).__name__}")
print(f"  - Classical: {type(classical_model).__name__}")

In [ ]:
def _resolve_extractor (model):

    return create_extractor (model).to (device)

classical_extractor =_resolve_extractor (classical_model)
quantum_extractor =_resolve_extractor (quantum_model)

print ("✓ Feature extractors created automatically!")
print (f"  - Classical extractor: {type (classical_extractor).__name__ }")
print (f"  - Quantum extractor: {type (quantum_extractor).__name__ }")

In [ ]:

loader =BreastMNISTLoader(batch_size =BATCH_SIZE ,image_size =IMAGE_SIZE)
train_loader ,test_loader ,val_loader =loader.get_dataloaders()

splits ={
"train":train_loader ,
"test":test_loader ,
"val":val_loader 
}

print(f"✓ Dataloaders prontos:")
print(f"  - Train batches: {len(train_loader)}")
print(f"  - Test batches: {len(test_loader)}")
print(f"  - Val batches: {len(val_loader)}")

In [ ]:
def extract_features (extractor ,dataloader ,output_dir ,split_name ,target_dim =None):
    os.makedirs (output_dir ,exist_ok =True )

    features_list =[]
    labels_list =[]

    extractor.eval ()
    with torch.no_grad ():
        for imgs ,labels in tqdm (dataloader ,desc =f"Extraindo {split_name }"):
            imgs =imgs.to (device )

            features =extractor (imgs )

            if target_dim is not None and features.shape [1 ]!=target_dim :

                features =features.view (features.size (0 ),-1 )
                features =torch.nn.functional.adaptive_avg_pool1d (
                features.unsqueeze (1 ),
                target_dim 
                ).squeeze (1 )

            features_list.append (features.cpu ().numpy ())
            labels_list.append (labels.cpu ().numpy ())

    features =np.concatenate (features_list ,axis =0 )
    labels =np.concatenate (labels_list ,axis =0 )

    np.save (os.path.join (output_dir ,f"{split_name }_features.npy"),features )
    np.save (os.path.join (output_dir ,f"{split_name }_labels.npy"),labels )

    print (f"  ✓ Saved: {split_name }_features.npy and {split_name }_labels.npy")

    return features.shape 

In [ ]:

classical_dir =os.path.join(OUTPUT_DIR ,"classical")
quantum_dir =os.path.join(OUTPUT_DIR ,"quantum")

os.makedirs(classical_dir ,exist_ok =True)
os.makedirs(quantum_dir ,exist_ok =True)

print(f"Saving features in: {OUTPUT_DIR }")
print(f"Target feature dim: {TARGET_FEATURE_DIM }\n")


for split_name ,split_loader in splits.items():
    print(f"\n{'='*60 }")
    print(f"Processing: {split_name.upper()}")
    print(f"{'='*60 }")


    print("\n[Classical]")
    c_shape =extract_features(classical_extractor ,split_loader ,classical_dir ,
    split_name ,target_dim =TARGET_FEATURE_DIM)
    print(f"  Shape: {c_shape }")


    print("\n[Quantum]")
    q_shape =extract_features(quantum_extractor ,split_loader ,quantum_dir ,
    split_name ,target_dim =TARGET_FEATURE_DIM)
    print(f"  Shape: {q_shape }")

print(f"\n\n{'='*60 }")
print("✓ EXTRACTION COMPLETE!")
print(f"{'='*60 }")
print(f"\nFiles saved in: {OUTPUT_DIR }")
if TARGET_FEATURE_DIM is not None :
    print(f"Feature dimension: {TARGET_FEATURE_DIM }")

In [ ]:

print("="*60)
print("EXTRACTED FEATURES SUMMARY")
print("="*60)

for split_name in ["train","val","test"]:
    print(f"\n{split_name.upper()} Split:")

    c_features =np.load(os.path.join(classical_dir ,f"{split_name }_features.npy"))
    q_features =np.load(os.path.join(quantum_dir ,f"{split_name }_features.npy"))
    labels =np.load(os.path.join(classical_dir ,f"{split_name }_labels.npy"))

    print(f"  Classical features: {c_features.shape }")
    print(f"  Quantum features:   {q_features.shape }")
    print(f"  Labels:             {labels.shape }")

print(f"\n{'='*60 }")
print("Files saved in:")
print(f"  Classical: {classical_dir }/")
print(f"  Quantum:   {quantum_dir }/")
print("="*60)